# Install required packages

In [1]:
! pip install transformers==4.56.1 sacremoses evaluate sacrebleu -q

# Imports

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer, MarianMTModel, MarianTokenizer, pipeline
import os
import torch
import evaluate
import gc
from kaggle_secrets import UserSecretsClient

2026-03-30 10:41:55.165526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774867315.188453    1027 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774867315.196098    1027 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774867315.215993    1027 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774867315.216011    1027 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774867315.216013    1027 computation_placer.cc:177] computation placer alr

# Load data

## Flores+
Covered languages: Nigerian Fulfulde (fuv).

In [3]:
user_secrets = UserSecretsClient()

# Set your HF token as an environment variable
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [4]:
fuv_flores_test = load_dataset("openlanguagedata/flores_plus", "fuv_Latn", split="devtest").to_pandas()
eng_flores_test = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest").to_pandas()

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

In [5]:
fuv_flores_test

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,fuv,Latn,nige1253,,“Jotta minɗo mari ɓera je wala diyabetis bawo ...,https://en.wikinews.org/wiki/Toronto_team-led_...,wikinews,"disease, research, canada",no,yes,1.0,devtest
1,1,fuv,Latn,nige1253,,"Dr. Ehud Ur, mallumjo manga ha jangirde lekki ...",https://en.wikinews.org/wiki/Toronto_team-led_...,wikinews,"disease, research, canada",no,yes,1.0,devtest
2,2,fuv,Latn,nige1253,,"Bana ɓe arti wawi sossay, o wala tabbaci ko ny...",https://en.wikinews.org/wiki/Toronto_team-led_...,wikinews,"disease, research, canada",no,yes,1.0,devtest
3,3,fuv,Latn,nige1253,,"Nyande Altin, Sra Deniyus. Kanko on mo hakkili...",https://en.wikinews.org/wiki/Nobel_Prize_in_Li...,wikinews,music,yes,yes,1.0,devtest
4,4,fuv,Latn,nige1253,,"Danius wi “jotta wala ko min waɗata, mi fiyi w...",https://en.wikinews.org/wiki/Nobel_Prize_in_Li...,wikinews,music,yes,yes,1.0,devtest
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007,1007,fuv,Latn,nige1253,,Baabe may ɗon yaaji wala yimɓe wonay jur ha to...,https://en.wikivoyage.org/wiki/Midnight_sun,wikivoyage,Natural wonders/Midnight sun,yes,no,1.0,devtest
1008,1008,fuv,Latn,nige1253,,Japanese ɗengal ɗon yaha ta'irde be ta'irde e ...,https://en.wikivoyage.org/wiki/Working_and_stu...,wikivoyage,Reason to travel/Working in Japan,yes,no,1.0,devtest
1009,1009,fuv,Latn,nige1253,,Tokgore kam jun on kare hani goɗɗo wata ha pel...,https://en.wikivoyage.org/wiki/Working_and_stu...,wikivoyage,Reason to travel/Working in Japan,yes,no,1.0,devtest
1010,1010,fuv,Latn,nige1253,,"Jonde jam ha baabal kugal ɗo mari nafu masin, ...",https://en.wikivoyage.org/wiki/Working_and_stu...,wikivoyage,Reason to travel/Working in Japan,yes,no,1.0,devtest


In [6]:
eng_flores_test

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,eng,Latn,stan1293,,"""We now have 4-month-old mice that are non-dia...",https://en.wikinews.org/wiki/Toronto_team-led_...,wikinews,"disease, research, canada",no,yes,1.0,devtest
1,1,eng,Latn,stan1293,,"Dr. Ehud Ur, professor of medicine at Dalhousi...",https://en.wikinews.org/wiki/Toronto_team-led_...,wikinews,"disease, research, canada",no,yes,1.0,devtest
2,2,eng,Latn,stan1293,,"Like some other experts, he is skeptical about...",https://en.wikinews.org/wiki/Toronto_team-led_...,wikinews,"disease, research, canada",no,yes,1.0,devtest
3,3,eng,Latn,stan1293,,"On Monday, Sara Danius, permanent secretary of...",https://en.wikinews.org/wiki/Nobel_Prize_in_Li...,wikinews,music,yes,yes,1.0,devtest
4,4,eng,Latn,stan1293,,"Danius said, ""Right now we are doing nothing. ...",https://en.wikinews.org/wiki/Nobel_Prize_in_Li...,wikinews,music,yes,yes,1.0,devtest
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007,1007,eng,Latn,stan1293,,"As the areas are sparsely populated, and light...",https://en.wikivoyage.org/wiki/Midnight_sun,wikivoyage,Natural wonders/Midnight sun,yes,no,1.0,devtest
1008,1008,eng,Latn,stan1293,,Japanese work culture is more hierarchical and...,https://en.wikivoyage.org/wiki/Working_and_stu...,wikivoyage,Reason to travel/Working in Japan,yes,no,1.0,devtest
1009,1009,eng,Latn,stan1293,,"Suits are standard business attire, and cowork...",https://en.wikivoyage.org/wiki/Working_and_stu...,wikivoyage,Reason to travel/Working in Japan,yes,no,1.0,devtest
1010,1010,eng,Latn,stan1293,,"Workplace harmony is crucial, emphasizing grou...",https://en.wikivoyage.org/wiki/Working_and_stu...,wikivoyage,Reason to travel/Working in Japan,yes,no,1.0,devtest


In [7]:
# Join the two dataframes on the 'id' column
flores_df = pd.merge(fuv_flores_test, eng_flores_test, on='id', suffixes=('_fuv', '_eng'))

# Only keep the 'text_eng' and 'text_fuv' columns
flores_df = flores_df[['text_eng', 'text_fuv']]

# Rename the columns to 'eng_Latn' and 'fuv_Latn'
flores_df.rename(columns={'text_eng': 'eng_Latn', 'text_fuv': 'fuv_Latn'}, inplace=True)

flores_df

,eng_Latn,fuv_Latn
0,"""We now have 4-month-old mice that are non-dia...",“Jotta minɗo mari ɓera je wala diyabetis bawo ...
1,"Dr. Ehud Ur, professor of medicine at Dalhousi...","Dr. Ehud Ur, mallumjo manga ha jangirde lekki ..."
2,"Like some other experts, he is skeptical about...","Bana ɓe arti wawi sossay, o wala tabbaci ko ny..."
3,"On Monday, Sara Danius, permanent secretary of...","Nyande Altin, Sra Deniyus. Kanko on mo hakkili..."
4,"Danius said, ""Right now we are doing nothing. ...","Danius wi “jotta wala ko min waɗata, mi fiyi w..."
...,...,...
1007,"As the areas are sparsely populated, and light...",Baabe may ɗon yaaji wala yimɓe wonay jur ha to...
1008,Japanese work culture is more hierarchical and...,Japanese ɗengal ɗon yaha ta'irde be ta'irde e ...
1009,"Suits are standard business attire, and cowork...",Tokgore kam jun on kare hani goɗɗo wata ha pel...
1010,"Workplace harmony is crucial, emphasizing grou...","Jonde jam ha baabal kugal ɗo mari nafu masin, ..."


## BOUQuET
Covered languages: Nigerian Fulfulde (fuv), Pulaar (fuc).

In [8]:
fuc_bouquet_test = load_dataset("facebook/bouquet", "fuc_Latn", split="test").to_pandas()
fuv_bouquet_test = load_dataset("facebook/bouquet", "fuv_Latn", split="test").to_pandas()

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

In [9]:
fuc_bouquet_test

,uniq_id,src_text,domain,par_comment,orig_text,tgt_text,newline_next,tags,register,par_id,split,level,src_lang,tgt_lang
0,P002-S1,Aɗe waawi waɗde maakoroni e maafe bekamel « pa...,how-to | instructions,"The second person pronoun ""you"" and verb conju...",ممكن تعملي مكرونة بالبشامل بسهولة,"You can easily make pasta ""Pastitsio"" with bec...",False,Semantic,uck,P002,test,sentence_level,fuc_Latn,eng_Latn
1,P002-S2,"Ko kaajeteɗaa ko heen fof ko ɗum maakoroni, te...",how-to | instructions,you 2nd person feminine gender used for implic...,كل اللي هتحتاجيه مكرونة ولحمة ومفرومة وبصل ولب...,"All you will need is pasta, ground beef, onion...",False,orthography,uck,P002,test,sentence_level,fuc_Latn,eng_Latn
2,P002-S3,"Njuɗaa teewu ngu wondude e basalle, pawa ɗum b...",how-to | instructions,It is a 2nd person verb conjugated in the femi...,شوحي اللحة المفرومة بالبصل وسيبيها على جمب,Sauté the ground beef with the onions and put ...,False,Lexical,uck,P002,test,sentence_level,fuc_Latn,eng_Latn
3,P002-S4,"Pasna maakoroni o, ngittaamo e jeyngol.",how-to | instructions,you 2nd person feminine gender used for reader.,اسلقي المكرونة وشيليها من عالنار,Boil the pasta and remove it from the heat.,False,Lexical,uck,P002,test,sentence_level,fuc_Latn,eng_Latn
4,P002-S5,"Pewna soos bekamel no leeliri, so ɓuuɓi tan, m...",how-to | instructions,you 2nd person feminine gender used for reader.,اعملي البشامل على مهلك وحطي عليه البيضة لما يبرد,Make the bechamel slowly and add the egg when ...,False,Lexical,uck,P002,test,sentence_level,fuc_Latn,eng_Latn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849,P462-S1,"Madam Hooreejo Suudu Sarɗiiji, Banndiraaɓe Rew...",other misc.,This paragraph represents the beginning of an ...,"Señora Presidenta, señoras y señores diputados...","Madam House Speaker, ladies and gentlemen Memb...",False,"sentence:fragment, miscellaneous:formula",usa,P462,test,sentence_level,fuc_Latn,eng_Latn
850,P462-S2,Miɗo ɗoo ɗaɓɓude woote hoolaare e ndeeɗoo suud...,other misc.,This paragraph represents the beginning of an ...,Comparezco ante esta Cámara al objeto de solic...,I am appearing before this House to request yo...,False,"miscellaneous:collocation, sentence:co-referen...",usa,P462,test,sentence_level,fuc_Latn,eng_Latn
851,P462-S3,"Njeñtudi woote ɗee ina laaɓti, ina hollita bal...",other misc.,This paragraph represents the beginning of an ...,Los resultados electorales han sido contundent...,The election results have been overwhelming an...,False,word:number,usa,P462,test,sentence_level,fuc_Latn,eng_Latn
852,P462-S4,Woote ɗe ɓiɗɓe leydi ndii kollitii e dow welli...,other misc.,This paragraph represents the beginning of an ...,Unas elecciones en las que los ciudadanos se h...,An election in which citizens have freely and ...,False,word:adverb,usa,P462,test,sentence_level,fuc_Latn,eng_Latn


In [10]:
fuv_bouquet_test

,uniq_id,src_text,domain,par_comment,orig_text,tgt_text,newline_next,tags,register,par_id,split,level,src_lang,tgt_lang
0,P002-S1,"""Aɗa waawi waɗde paasta e nofru """"Pastitsio"""" ...",how-to | instructions,"The second person pronoun ""you"" and verb conju...",ممكن تعملي مكرونة بالبشامل بسهولة,"You can easily make pasta ""Pastitsio"" with bec...",False,Semantic,uck,P002,test,sentence_level,fuv_Latn,eng_Latn
1,P002-S2,"Ko njiɗɗaa fof ko paasta, na’i dowri, njuumri,...",how-to | instructions,you 2nd person feminine gender used for implic...,كل اللي هتحتاجيه مكرونة ولحمة ومفرومة وبصل ولب...,"All you will need is pasta, ground beef, onion...",False,orthography,uck,P002,test,sentence_level,fuv_Latn,eng_Latn
2,P002-S3,"Sauuté 6uu6ol les ngol e tineeje cfee, 6uucca ...",how-to | instructions,It is a 2nd person verb conjugated in the femi...,شوحي اللحة المفرومة بالبصل وسيبيها على جمب,Sauté the ground beef with the onions and put ...,False,Lexical,uck,P002,test,sentence_level,fuv_Latn,eng_Latn
3,P002-S4,"Buuɓnu paasta oo, itta ɗum e nguleeki.",how-to | instructions,you 2nd person feminine gender used for reader.,اسلقي المكرونة وشيليها من عالنار,Boil the pasta and remove it from the heat.,False,Lexical,uck,P002,test,sentence_level,fuv_Latn,eng_Latn
4,P002-S5,"Mbaɗaa beechamel seese-seese, ɓeydu egg oo so ...",how-to | instructions,you 2nd person feminine gender used for reader.,اعملي البشامل على مهلك وحطي عليه البيضة لما يبرد,Make the bechamel slowly and add the egg when ...,False,Lexical,uck,P002,test,sentence_level,fuv_Latn,eng_Latn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849,P462-S1,"Madam Wolwoowo Saare, rowɓe be worɓe Laatiɓe n...",other misc.,This paragraph represents the beginning of an ...,"Señora Presidenta, señoras y señores diputados...","Madam House Speaker, ladies and gentlemen Memb...",False,"sentence:fragment, miscellaneous:formula",usa,P462,test,sentence_level,fuv_Latn,eng_Latn
850,P462-S2,Mi daro ha yeeso Saare ɗo ngam ɗaɓɓutuki yerdu...,other misc.,This paragraph represents the beginning of an ...,Comparezco ante esta Cámara al objeto de solic...,I am appearing before this House to request yo...,False,"miscellaneous:collocation, sentence:co-referen...",usa,P462,test,sentence_level,fuv_Latn,eng_Latn
851,P462-S3,Ngasnaaɗi cuɓol mai laatake ɗuɗɗum masin nden ...,other misc.,This paragraph represents the beginning of an ...,Los resultados electorales han sido contundent...,The election results have been overwhelming an...,False,word:number,usa,P462,test,sentence_level,fuv_Latn,eng_Latn
852,P462-S4,Cuɓol jei ha himɓe lesdi heɓi holluki be hoore...,other misc.,This paragraph represents the beginning of an ...,Unas elecciones en las que los ciudadanos se h...,An election in which citizens have freely and ...,False,word:adverb,usa,P462,test,sentence_level,fuv_Latn,eng_Latn


In [11]:
# Rename "src_text"
fuc_bouquet_test.rename(columns={'src_text': 'fuc_Latn', 'tgt_text': 'eng_Latn'}, inplace=True)
fuv_bouquet_test.rename(columns={'src_text': 'fuv_Latn'}, inplace=True)

# Merge the two dataframes on the "uniq_id" column
bouquet_df = pd.merge(fuc_bouquet_test, fuv_bouquet_test, on='uniq_id')

# Only keep the "fuc_Latn", "fuv_Latn", and "eng_Latn" columns
bouquet_df = bouquet_df[['fuc_Latn', 'fuv_Latn', 'eng_Latn']]

bouquet_df

,fuc_Latn,fuv_Latn,eng_Latn
0,Aɗe waawi waɗde maakoroni e maafe bekamel « pa...,"""Aɗa waawi waɗde paasta e nofru """"Pastitsio"""" ...","You can easily make pasta ""Pastitsio"" with bec..."
1,"Ko kaajeteɗaa ko heen fof ko ɗum maakoroni, te...","Ko njiɗɗaa fof ko paasta, na’i dowri, njuumri,...","All you will need is pasta, ground beef, onion..."
2,"Njuɗaa teewu ngu wondude e basalle, pawa ɗum b...","Sauuté 6uu6ol les ngol e tineeje cfee, 6uucca ...",Sauté the ground beef with the onions and put ...
3,"Pasna maakoroni o, ngittaamo e jeyngol.","Buuɓnu paasta oo, itta ɗum e nguleeki.",Boil the pasta and remove it from the heat.
4,"Pewna soos bekamel no leeliri, so ɓuuɓi tan, m...","Mbaɗaa beechamel seese-seese, ɓeydu egg oo so ...",Make the bechamel slowly and add the egg when ...
...,...,...,...
849,"Madam Hooreejo Suudu Sarɗiiji, Banndiraaɓe Rew...","Madam Wolwoowo Saare, rowɓe be worɓe Laatiɓe n...","Madam House Speaker, ladies and gentlemen Memb..."
850,Miɗo ɗoo ɗaɓɓude woote hoolaare e ndeeɗoo suud...,Mi daro ha yeeso Saare ɗo ngam ɗaɓɓutuki yerdu...,I am appearing before this House to request yo...
851,"Njeñtudi woote ɗee ina laaɓti, ina hollita bal...",Ngasnaaɗi cuɓol mai laatake ɗuɗɗum masin nden ...,The election results have been overwhelming an...
852,Woote ɗe ɓiɗɓe leydi ndii kollitii e dow welli...,Cuɓol jei ha himɓe lesdi heɓi holluki be hoore...,An election in which citizens have freely and ...


## UDHR
Covered languages: Dagbani (dag).

In [12]:
# Load the UDHR (Universal Declaration of Human Rights) dataset for Dagbani (dag)
udhr_dag_url = "http://efele.net/udhr/d/udhr_dag.xml"
udhr_eng_url = "http://efele.net/udhr/d/udhr_eng.xml"

def load_udhr(url):
    if os.path.exists(url.split("/")[-1]):
        with open(url.split("/")[-1], "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "xml")
    else:
        response = requests.get(url)
        # Download the file and save it locally for future use
        with open(url.split("/")[-1], "w", encoding="utf-8") as f:
            f.write(response.text)
        soup = BeautifulSoup(response.content, "xml")
    text = [p.get_text() for p in soup.find_all("para")]
    return text

In [13]:
udhr_eng = load_udhr(udhr_eng_url)

# Merge English paragraphs 1 through 7
udhr_eng_para_1 = " ".join(udhr_eng[:7])

# Merge English paragraphs 8 through 10
udhr_eng_para_2 = " ".join(udhr_eng[7:10])

udhr_eng = [udhr_eng_para_1, udhr_eng_para_2] + udhr_eng[10:]

In [14]:
udhr_dag = load_udhr(udhr_dag_url)

# Merge Dagbani paragraphs 1 through 3
udhr_dag_para_1 = " ".join(udhr_dag[:3])

# Merge Dagbani paragraphs 4 through 8
udhr_dag_para_2 = " ".join(udhr_dag[3:8])

udhr_dag = [udhr_dag_para_1, udhr_dag_para_2] + udhr_dag[8:]

In [15]:
assert len(udhr_dag) == len(udhr_eng), f"The number of paragraphs in Dagbani and English UDHR should be the same. Found {len(udhr_dag)} and {len(udhr_eng)} respectively."

In [16]:
# Convert to Pandas DataFrame
udhr_df = pd.DataFrame({
    "eng_Latn": udhr_eng,
    "dag_Latn": udhr_dag
})

udhr_df

,eng_Latn,dag_Latn
0,Whereas recognition of the inherent dignity an...,"Yɛlimaŋli, bɛhigu kam yi yɛn mali kɔre ni dari..."
1,"Now, therefore, The General Assembly Proclaims...","Dinzugu, pumpɔŋɔ, Jɛneral Asɛmbli Laɣiŋgu ni d..."
2,All human beings are born free and equal in di...,"Sal' la sala. Bɛhig' be sokam sanimi, din pa l..."
3,Everyone is entitled to all the rights and fre...,Zaligu dinkam be litaafi ŋɔ ni nyɛla din maani...
4,"Furthermore, no distinction shall be made on t...",Din nyaaŋa di ka soli ka so ʒem o kpee ni o ti...
5,"Everyone has the right to life, liberty and th...",So di pirim o wɔɣiritali zuɣu yɛli ni o tum tu...
6,No one shall be held in slavery or servitude; ...,Zaligu bi lan n saɣi ni so zaŋ o ygunaadam kpe...
7,No one shall be subjected to torture or to cru...,Zaligu bi lan saɣi ni bɛ niŋ so alaka bee fuku...
8,Everyone has the right to recognition everywhe...,Zaŋ n chaŋ dunia-yili lɔɔ bee dunia-yili zalig...
9,All are equal before the law and are entitled ...,"Zaŋ n chaŋ zal' shɛŋa ŋan be kundi puuni, yɛll..."


# Preprocess Data

In [17]:
# this code is adapted from  the Stopes repo of the NLLB team
# https://github.com/facebookresearch/stopes/blob/main/stopes/pipelines/monolingual/monolingual_line_processor.py#L214

import re
import sys
import typing as tp
import unicodedata
from sacremoses import MosesPunctNormalizer


mpn = MosesPunctNormalizer(lang="en")
mpn.substitutions = [
    (re.compile(r), sub) for r, sub in mpn.substitutions
]


def get_non_printing_char_replacer(replace_by: str = " ") -> tp.Callable[[str], str]:
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        # same as \p{C} in perl
        # see https://www.unicode.org/reports/tr44/#General_Category_Values
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }

    def replace_non_printing_char(line) -> str:
        return line.translate(non_printable_map)

    return replace_non_printing_char

replace_nonprint = get_non_printing_char_replacer(" ")

def preproc(text):
    clean = mpn.normalize(text)
    clean = replace_nonprint(clean)
    # replace 𝓕𝔯𝔞𝔫𝔠𝔢𝔰𝔠𝔞 by Francesca
    clean = unicodedata.normalize("NFKC", clean)
    return clean

In [18]:
# Get the vocabulary of one dataset to check if the preprocessing is working correctly
from collections import Counter
print("Before preprocessing:")
count_before = Counter("".join(flores_df['eng_Latn']))
print("Flores English vocab size:", len(count_before))

datasets = {
    'flores': flores_df,
    'bouquet': bouquet_df,
    'udhr': udhr_df,
    # 'pontoon': pontoon_df
}

# Apply preprocessing to all datasets
for name, df in datasets.items():
    print(f"Processing {name} dataset...")
    for col in df.columns:
        df[col] = df[col].apply(preproc)

print("\nAfter preprocessing:")
count_after = Counter("".join(flores_df['eng_Latn']))
print("Flores English vocab size:", len(count_after))

for char in count_before:
    if char not in count_after:
        print(f"Character '{char}' (freq: {count_before[char]}) was removed during preprocessing.")

Before preprocessing:
Flores English vocab size: 106
Processing flores dataset...
Processing bouquet dataset...
Processing udhr dataset...

After preprocessing:
Flores English vocab size: 97
Character '—' (freq: 10) was removed during preprocessing.
Character '–' (freq: 4) was removed during preprocessing.
Character '’' (freq: 14) was removed during preprocessing.
Character '¾' (freq: 1) was removed during preprocessing.
Character '½' (freq: 1) was removed during preprocessing.
Character '“' (freq: 9) was removed during preprocessing.
Character '‘' (freq: 1) was removed during preprocessing.
Character '”' (freq: 9) was removed during preprocessing.
Character '​' (freq: 2) was removed during preprocessing.
Character '²' (freq: 2) was removed during preprocessing.


# Evaluation

In [19]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
ter_metric = evaluate.load("ter")

def metrics_calc(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    bleu_result = bleu_metric.compute(predictions = preds, references = labels)
    spm_result = bleu_metric.compute(predictions = preds, references = labels, tokenize='flores101')
    chrf_result = chrf_metric.compute(predictions = preds, references = labels, word_order=2)
    ter_result = ter_metric.compute(predictions = preds, references = labels)

    result = {
        'bleu': bleu_result['score'],
        'spbleu': spm_result['score'],
        'chrf++': chrf_result['score'],
        'ter': ter_result['score']
    }
    result = {k: round(v, 2) for k, v in result.items()}

    return result

In [20]:
def evaluate_mt_model(model_name, inference_fn, data, **kwargs):
    results = []

    for dataset_name, df in data.items():
        print(f"Evaluating on {dataset_name} dataset...")
        langs = df.columns.tolist()
        langs.pop(langs.index('eng_Latn'))
        eng_texts = df['eng_Latn'].tolist()
        for lang in langs:
            # From English to the target language
            print(f"Translating from English to {lang}...")
            translations = inference_fn(model_name, eng_texts, "eng_Latn", lang, **kwargs)
            if translations is None:
                continue
            references = df[lang].tolist()
            from_metrics = metrics_calc(translations, references)
            from_prefix = f"eng_Latn-{lang}"
            from_metrics = {f"{from_prefix}_{k}": v for k, v in from_metrics.items()}
            print(f"Results for English to {lang}: {from_metrics}")

            # From the target language to English
            print(f"Translating from {lang} to English...")
            translations = inference_fn(model_name, references, lang, "eng_Latn", **kwargs)
            if translations is None:
                continue
            to_metrics = metrics_calc(translations, eng_texts)
            to_prefix = f"{lang}-eng_Latn"
            to_metrics = {f"{to_prefix}_{k}": v for k, v in to_metrics.items()}
            print(f"Results for {lang} to English: {to_metrics}\n")

            results.append({
                'model': model_name,
                'dataset': dataset_name,
                'language_pair': f"eng_Latn-{lang}",
                **from_metrics,
                **to_metrics
                # 'from_metrics': from_metrics,
                # 'to_metrics': to_metrics
            })
    # return results
    return pd.DataFrame(results)

## NLLB-200 distilled 600M
Supported languages: Nigerian Fulfulde (fuv).

In [21]:
NLLB_600M_MODEL_ID = "facebook/nllb-200-distilled-600M"

nllb_600m_tokenizer = AutoTokenizer.from_pretrained(NLLB_600M_MODEL_ID)

# Get the list of supported languages for the NLLB models
NLLB_SUPPORTED_LANGS = nllb_600m_tokenizer.special_tokens_map['additional_special_tokens']

In [22]:
def nllb_inference(model_name, src_texts, src_lang, tgt_lang, **kwargs):
    # Check if the model supports the source and target languages
    if src_lang not in NLLB_SUPPORTED_LANGS:
        print(f"Source language '{src_lang}' is not supported by the model.")
        return None
    if tgt_lang not in NLLB_SUPPORTED_LANGS:
        print(f"Target language '{tgt_lang}' is not supported by the model.")
        return None

    # Use pipeline
    translation_pipeline = pipeline(
        "translation",
        model=model_name,
        src_lang=src_lang,
        tgt_lang=tgt_lang,
        device=0 if torch.cuda.is_available() else -1,
        **kwargs
    )

    results = translation_pipeline(src_texts)

    return [res['translation_text'] for res in results]

In [23]:
# Free up GPU memory
gc.collect()
torch.cuda.empty_cache()

In [24]:
# nllb_600m_results = evaluate_mt_model(NLLB_600M_MODEL_ID, nllb_inference, datasets, batch_size=16)
# nllb_600m_results

## NLLB-200 3B
Supported languages: Nigerian Fulfulde (fuv).

## OPUS-MT

In [25]:
OPUS_MODEL_NAME = "Helsinki-NLP/opus-mt-tc-bible-big-mul-mul"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

"""
OPUS_TOKENIZER = MarianTokenizer.from_pretrained(OPUS_MODEL_NAME)
OPUS_MODEL = MarianMTModel.from_pretrained(OPUS_MODEL_NAME).to(device)
"""

'\nOPUS_TOKENIZER = MarianTokenizer.from_pretrained(OPUS_MODEL_NAME)\nOPUS_MODEL = MarianMTModel.from_pretrained(OPUS_MODEL_NAME).to(device)\n'

In [26]:
src_text = [
    ">>kab<< You'd better not speak to Tom about that.",
    ">>kab<< How are you?"
]

In [27]:
"""
inputs = OPUS_TOKENIZER(src_text, return_tensors="pt", padding=True).to(OPUS_MODEL.device)
translated = OPUS_MODEL.generate(**inputs)

for t in translated:
    print(OPUS_TOKENIZER.decode(t, skip_special_tokens=True))
"""

'\ninputs = OPUS_TOKENIZER(src_text, return_tensors="pt", padding=True).to(OPUS_MODEL.device)\ntranslated = OPUS_MODEL.generate(**inputs)\n\nfor t in translated:\n    print(OPUS_TOKENIZER.decode(t, skip_special_tokens=True))\n'

In [28]:
OPUS_PIPELINE = pipeline("translation", model="Helsinki-NLP/opus-mt-tc-bible-big-mul-mul", device=device)

Device set to use cuda


In [29]:
print(OPUS_PIPELINE(src_text, batch_size=2))

[{'translation_text': '<0xE1><0xB8><0xA4>ur aț-țmeslayeḍ i Tom ɣef wayagi.'}, {'translation_text': 'Amek tella?'}]


In [30]:
def marian_inference(model_name, src_texts, src_lang, tgt_lang, **kwargs):
    # Prepend language tags to the source texts
    tgt_lang = tgt_lang.split('_')[0]
    src_texts = [f">>{tgt_lang}<< {text}" for text in src_texts]

    results = OPUS_PIPELINE(src_texts, **kwargs)

    return [res['translation_text'] for res in results]

In [32]:
opus_mt_results = evaluate_mt_model(OPUS_MODEL_NAME, marian_inference, datasets, batch_size=64)
opus_mt_results

Evaluating on flores dataset...
Translating from English to fuv_Latn...
Results for English to fuv_Latn: {'eng_Latn-fuv_Latn_bleu': 0.66, 'eng_Latn-fuv_Latn_spbleu': 0.77, 'eng_Latn-fuv_Latn_chrf++': 7.2, 'eng_Latn-fuv_Latn_ter': 113.56}
Translating from fuv_Latn to English...
Results for fuv_Latn to English: {'fuv_Latn-eng_Latn_bleu': 2.54, 'fuv_Latn-eng_Latn_spbleu': 3.34, 'fuv_Latn-eng_Latn_chrf++': 18.53, 'fuv_Latn-eng_Latn_ter': 151.79}

Evaluating on bouquet dataset...
Translating from English to fuc_Latn...
Results for English to fuc_Latn: {'eng_Latn-fuc_Latn_bleu': 0.98, 'eng_Latn-fuc_Latn_spbleu': 1.13, 'eng_Latn-fuc_Latn_chrf++': 9.92, 'eng_Latn-fuc_Latn_ter': 143.37}
Translating from fuc_Latn to English...
Results for fuc_Latn to English: {'fuc_Latn-eng_Latn_bleu': 1.09, 'fuc_Latn-eng_Latn_spbleu': 1.56, 'fuc_Latn-eng_Latn_chrf++': 15.49, 'fuc_Latn-eng_Latn_ter': 157.27}

Translating from English to fuv_Latn...
Results for English to fuv_Latn: {'eng_Latn-fuv_Latn_bleu': 0.88

Your input_length: 492 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Results for English to dag_Latn: {'eng_Latn-dag_Latn_bleu': 0.05, 'eng_Latn-dag_Latn_spbleu': 0.02, 'eng_Latn-dag_Latn_chrf++': 8.37, 'eng_Latn-dag_Latn_ter': 111.27}
Translating from dag_Latn to English...
Results for dag_Latn to English: {'dag_Latn-eng_Latn_bleu': 0.24, 'dag_Latn-eng_Latn_spbleu': 0.21, 'dag_Latn-eng_Latn_chrf++': 12.41, 'dag_Latn-eng_Latn_ter': 175.67}



,model,dataset,language_pair,eng_Latn-fuv_Latn_bleu,eng_Latn-fuv_Latn_spbleu,eng_Latn-fuv_Latn_chrf++,eng_Latn-fuv_Latn_ter,fuv_Latn-eng_Latn_bleu,fuv_Latn-eng_Latn_spbleu,fuv_Latn-eng_Latn_chrf++,...,fuc_Latn-eng_Latn_chrf++,fuc_Latn-eng_Latn_ter,eng_Latn-dag_Latn_bleu,eng_Latn-dag_Latn_spbleu,eng_Latn-dag_Latn_chrf++,eng_Latn-dag_Latn_ter,dag_Latn-eng_Latn_bleu,dag_Latn-eng_Latn_spbleu,dag_Latn-eng_Latn_chrf++,dag_Latn-eng_Latn_ter
0,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,flores,eng_Latn-fuv_Latn,0.66,0.77,7.20,113.56,2.54,3.34,18.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,bouquet,eng_Latn-fuc_Latn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.49,157.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,bouquet,eng_Latn-fuv_Latn,0.88,0.50,5.48,136.01,2.47,3.08,17.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,udhr,eng_Latn-dag_Latn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.05,0.02,8.37,111.27,0.24,0.21,12.41,175.67
